In [ ]:
import os
import numpy as np
import laspy
from glob import glob
from tqdm import tqdm

In [ ]:
ORIGINAL_SCENE_DIR = ""
RAW_PREDICTIONS_DIR = ""
OUTPUT_DIR          = ""
NUM_CLASSES = 3

In [ ]:
def stitch_semantic_results():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    scene_files = glob(os.path.join(ORIGINAL_SCENE_DIR, "*.la*"))
    
    for scene_path in scene_files:
        scene_name = os.path.splitext(os.path.basename(scene_path))[0]
        
        las = laspy.read(scene_path)
        num_points = len(las.points)
        
        logits_sum = np.zeros((num_points, NUM_CLASSES), dtype=np.float32)
        counts = np.zeros(num_points, dtype=np.int32)
        
        patch_files = glob(os.path.join(RAW_PREDICTIONS_DIR, f"{scene_name}*.npz"))
        
        if not patch_files:
            continue

        for patch_path in tqdm(patch_files, desc=f"Stitching {scene_name}"):
            data = np.load(patch_path)
            indices = data['global_indices']
            logits = data['semantic_logits']
            
            np.add.at(logits_sum, indices, logits)
            np.add.at(counts, indices, 1)
            
        mask = counts > 0
        if not np.any(mask):
            continue
            
        final_logits = logits_sum[mask] / counts[mask, None]
        final_labels = np.argmax(final_logits, axis=1)
        
        out_las = laspy.LasData(las.header)
        out_las.points = las.points[mask]
        
        out_las.add_extra_dim(laspy.ExtraBytesParams(name="ai_prediction", type=np.int32))
        out_las.ai_prediction = final_labels
        
        output_path = os.path.join(OUTPUT_DIR, f"{scene_name}_stitched.laz")
        out_las.write(output_path)

if __name__ == "__main__":
    stitch_semantic_results()